# 01 — FPL Data Collection

Collects raw data from the FPL official API and saves it as Parquet files.

**Endpoints used:**
- `/bootstrap-static/` — all players, teams, gameweek info
- `/fixtures/` — all fixtures + fixture difficulty ratings (FDR)
- `/element-summary/{id}/` — per-player gameweek history

**Outputs:**
- `data/raw/fpl_players.parquet` — one row per player, current season snapshot
- `data/raw/fpl_gameweeks.parquet` — one row per player per gameweek
- `data/raw/fpl_fixtures.parquet` — all fixtures with FDR
- `data/raw/fpl_teams.parquet` — team ID to name mapping

## 0. Smoke test — verify API is reachable

In [1]:
import requests

BASE_URL = "https://fantasy.premierleague.com/api/"

r = requests.get(f"{BASE_URL}bootstrap-static/")
r.raise_for_status()
data = r.json()

print("Status:", r.status_code)
print("Top-level keys:", list(data.keys()))
print("Player count:", len(data["elements"]))
print("Team count:", len(data["teams"]))
print("Total gameweeks:", len(data["events"]))

Status: 200
Top-level keys: ['chips', 'events', 'game_settings', 'game_config', 'phases', 'teams', 'total_players', 'element_stats', 'element_types', 'elements']
Player count: 825
Team count: 20
Total gameweeks: 38


## 1. Imports & setup

In [2]:
import time
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw")
RAW.mkdir(parents=True, exist_ok=True)

## 2. Bootstrap — players, teams, gameweek metadata

In [3]:
bootstrap = requests.get(f"{BASE_URL}bootstrap-static/").json()

# --- Players (elements) ---
players_df = pd.DataFrame(bootstrap["elements"])
print(players_df.shape)
players_df.head()

(825, 105)


,can_transact,can_select,chance_of_playing_next_round,chance_of_playing_this_round,code,cost_change_event,cost_change_event_fall,cost_change_start,cost_change_start_fall,price_change_percent,...,now_cost_rank_type,form_rank,form_rank_type,points_per_game_rank,points_per_game_rank_type,selected_rank,selected_rank_type,starts_per_90,clean_sheets_per_90,defensive_contribution_per_90
0,True,True,NaN,NaN,154561,0,0,5,-5,0,...,1,24,2,51,3,9,1,1.0,0.48,0.00
1,True,True,NaN,NaN,109745,0,0,-5,5,0,...,43,406,53,581,66,287,37,0.0,0.00,0.00
2,True,False,0.0,0.0,463748,0,0,0,0,0,...,53,424,63,598,75,373,54,0.0,0.00,0.00
3,True,True,NaN,NaN,551221,0,0,-1,1,0,...,89,393,46,570,60,362,52,0.0,0.00,0.00
4,True,True,75.0,100.0,226597,0,0,12,-12,0,...,1,11,7,2,1,5,1,1.0,0.58,9.31


In [4]:
# Columns we care about at this stage
PLAYER_COLS = [
    "id", "first_name", "second_name", "web_name",
    "element_type",   # 1=GKP, 2=DEF, 3=MID, 4=FWD
    "team",           # team ID
    "now_cost",       # price x 10 (e.g. 65 = £6.5m)
    "selected_by_percent",
    "total_points",
    "minutes",
    "goals_scored", "assists", "clean_sheets", "bonus",
    "ict_index",
    "form",
    "status",         # a=available, d=doubtful, i=injured, s=suspended, u=unavailable
]

players_df = players_df[PLAYER_COLS].copy()
players_df["now_cost"] = players_df["now_cost"] / 10  # convert to £m
players_df.head()

,id,first_name,second_name,web_name,element_type,team,now_cost,selected_by_percent,total_points,minutes,goals_scored,assists,clean_sheets,bonus,ict_index,form,status
0,1,David,Raya Martín,Raya,1,1,6.0,34.7,129,2790,0,0,15,6,48.0,7.0,a
1,2,Kepa,Arrizabalaga Revuelta,Arrizabalaga,1,1,4.0,0.4,0,0,0,0,0,0,0.0,0.0,a
2,3,Karl,Hein,Hein,1,1,4.0,0.2,0,0,0,0,0,0,0.0,0.0,u
3,4,Tommy,Setford,Setford,1,1,3.9,0.2,0,0,0,0,0,0,0.0,0.0,a
4,5,Gabriel,dos Santos Magalhães,Gabriel,2,1,7.2,42.9,173,2165,3,4,14,25,99.2,9.0,d


In [5]:
# --- Teams ---
teams_df = pd.DataFrame(bootstrap["teams"])[["id", "name", "short_name"]]
teams_df.head()

,id,name,short_name
0,1,Arsenal,ARS
1,2,Aston Villa,AVL
2,3,Burnley,BUR
3,4,Bournemouth,BOU
4,5,Brentford,BRE


In [6]:
# --- Gameweek metadata ---
events_df = pd.DataFrame(bootstrap["events"])[
    ["id", "name", "deadline_time", "finished", "is_current", "is_next",
     "average_entry_score", "highest_score"]
]
current_gw = events_df.loc[events_df["is_current"], "id"].values
print("Current gameweek:", current_gw)
events_df.head()

Current gameweek: [31]


,id,name,deadline_time,finished,is_current,is_next,average_entry_score,highest_score
0,1,Gameweek 1,2025-08-15T17:30:00Z,True,False,False,54,127.0
1,2,Gameweek 2,2025-08-22T17:30:00Z,True,False,False,51,140.0
2,3,Gameweek 3,2025-08-30T10:00:00Z,True,False,False,48,118.0
3,4,Gameweek 4,2025-09-13T10:00:00Z,True,False,False,63,139.0
4,5,Gameweek 5,2025-09-20T10:00:00Z,True,False,False,42,112.0


## 3. Fixtures

In [7]:
fixtures_raw = requests.get(f"{BASE_URL}fixtures/").json()
fixtures_df = pd.DataFrame(fixtures_raw)

FIXTURE_COLS = [
    "id", "event",           # gameweek number
    "team_h", "team_a",
    "team_h_difficulty", "team_a_difficulty",  # FDR (1-5)
    "team_h_score", "team_a_score",
    "finished", "kickoff_time",
]

fixtures_df = fixtures_df[FIXTURE_COLS].copy()
print(fixtures_df.shape)
fixtures_df.head()

(380, 10)


,id,event,team_h,team_a,team_h_difficulty,team_a_difficulty,team_h_score,team_a_score,finished,kickoff_time
0,307,NaN,13,8,3,5,NaN,NaN,False,None
1,1,1.0,12,4,3,4,4.0,2.0,True,2025-08-15T19:00:00Z
2,2,1.0,2,15,3,4,0.0,0.0,True,2025-08-16T11:30:00Z
3,3,1.0,6,10,2,3,1.0,1.0,True,2025-08-16T14:00:00Z
4,6,1.0,18,3,1,3,3.0,0.0,True,2025-08-16T14:00:00Z


## 4. Per-player gameweek history

One request per player — ~700 requests. A small delay is added between calls to avoid hammering the API.

In [8]:
# WARNING: takes ~5-10 minutes for the full player list. Run once and cache to Parquet.

all_gw_records = []
player_ids = players_df["id"].tolist()

for i, pid in enumerate(player_ids):
    url = f"{BASE_URL}element-summary/{pid}/"
    resp = requests.get(url)
    if resp.status_code != 200:
        print(f"  Skipped player {pid} — status {resp.status_code}")
        continue

    for row in resp.json().get("history", []):
        row["player_id"] = pid
        all_gw_records.append(row)

    if i % 50 == 0:
        print(f"  {i}/{len(player_ids)} players fetched...")

    time.sleep(0.05)  # 50ms between requests

gw_df = pd.DataFrame(all_gw_records)
print("Total gameweek rows:", len(gw_df))
gw_df.head()

  0/825 players fetched...
  50/825 players fetched...
  100/825 players fetched...
  150/825 players fetched...
  200/825 players fetched...
  250/825 players fetched...
  300/825 players fetched...
  350/825 players fetched...
  400/825 players fetched...
  450/825 players fetched...
  500/825 players fetched...
  550/825 players fetched...
  600/825 players fetched...
  650/825 players fetched...
  700/825 players fetched...
  750/825 players fetched...
  800/825 players fetched...
Total gameweek rows: 23829


,element,fixture,opponent_team,total_points,was_home,kickoff_time,team_h_score,team_a_score,round,modified,...,expected_goals,expected_assists,expected_goal_involvements,expected_goals_conceded,value,transfers_balance,selected,transfers_in,transfers_out,player_id
0,1,9,14,10,False,2025-08-17T15:30:00Z,0,1,1,False,...,0.00,0.00,0.00,1.52,55,0,1531911,0,0,1
1,1,11,11,6,True,2025-08-23T16:30:00Z,5,0,2,False,...,0.00,0.00,0.00,0.17,55,218659,2284634,277339,58680,1
2,1,25,12,2,False,2025-08-31T15:30:00Z,1,0,3,False,...,0.00,0.02,0.02,0.52,55,-12311,2406964,146739,159050,1
3,1,31,16,6,True,2025-09-13T11:30:00Z,3,0,4,False,...,0.00,0.00,0.00,0.20,55,171289,2765759,289041,117752,1
4,1,41,13,2,True,2025-09-21T15:30:00Z,1,1,5,False,...,0.00,0.01,0.01,0.89,55,-9786,2762632,98100,107886,1


In [9]:
GW_COLS = [
    "player_id", "round",
    "total_points",
    "minutes", "goals_scored", "assists", "clean_sheets",
    "goals_conceded", "own_goals", "penalties_saved", "penalties_missed",
    "yellow_cards", "red_cards", "saves", "bonus", "bps",
    "influence", "creativity", "threat", "ict_index",
    "value",       # price at time of this gameweek (x 10)
    "selected",    # ownership count at time of this gameweek
    "was_home",
    "opponent_team",
    "kickoff_time",
]

gw_df = gw_df[[c for c in GW_COLS if c in gw_df.columns]].copy()
gw_df["value"] = gw_df["value"] / 10
gw_df.head()

,player_id,round,total_points,minutes,goals_scored,assists,clean_sheets,goals_conceded,own_goals,penalties_saved,...,bps,influence,creativity,threat,ict_index,value,selected,was_home,opponent_team,kickoff_time
0,1,1,10,90,0,0,1,0,0,0,...,38,49.2,0.0,0.0,4.9,5.5,1531911,False,14,2025-08-17T15:30:00Z
1,1,2,6,90,0,0,1,0,0,0,...,28,13.4,0.0,0.0,1.3,5.5,2284634,True,11,2025-08-23T16:30:00Z
2,1,3,2,90,0,0,0,1,0,0,...,12,20.0,10.0,0.0,3.0,5.5,2406964,False,12,2025-08-31T15:30:00Z
3,1,4,6,90,0,0,1,0,0,0,...,24,12.8,0.0,0.0,1.3,5.5,2765759,True,16,2025-09-13T11:30:00Z
4,1,5,2,90,0,0,0,1,0,0,...,13,21.4,0.0,0.0,2.1,5.5,2762632,True,13,2025-09-21T15:30:00Z


## 5. Save to Parquet

In [10]:
players_df.to_parquet(RAW / "fpl_players.parquet", index=False)
fixtures_df.to_parquet(RAW / "fpl_fixtures.parquet", index=False)
gw_df.to_parquet(RAW / "fpl_gameweeks.parquet", index=False)
teams_df.to_parquet(RAW / "fpl_teams.parquet", index=False)

print("Saved:")
for name, df in [
    ("fpl_players.parquet", players_df),
    ("fpl_fixtures.parquet", fixtures_df),
    ("fpl_gameweeks.parquet", gw_df),
    ("fpl_teams.parquet", teams_df),
]:
    print(f"  {name:<30} {len(df):>5} rows")

Saved:
  fpl_players.parquet              825 rows
  fpl_fixtures.parquet             380 rows
  fpl_gameweeks.parquet          23829 rows
  fpl_teams.parquet                 20 rows
